# River flood forest restoration avoided EADs and bauxite mining risk

This notebook quantifies how much of the river-flood forest restoration area that provides positive avoided EAD is located on mapped bauxite reserves, and how much of that bauxite-overlapping restoration benefit area is protected.

The analysis uses the river-flood avoided-EAD rasters from DPhil paper 2, converted from JMD to USD using `1/150`, and rasterizes bauxite reserves and protected areas onto the same 30 m EPSG:3448 grid. Restoration benefit area is defined as pixels with positive avoided EAD in either the minimum or maximum river-flood scenario.

In [ ]:
from pathlib import Path
import sys

BASE = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
ROBYN_LIBRARY_PATH = BASE / "robyns_libraries"
if str(ROBYN_LIBRARY_PATH) not in sys.path:
    sys.path.append(str(ROBYN_LIBRARY_PATH))

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import rasterio
import Robyn_paper_2_defs
from IPython.display import Markdown, display
from matplotlib.colors import BoundaryNorm, ListedColormap
from rasterio.features import rasterize

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

In [ ]:
PAPER2 = BASE / "dphil_paper_2"
PAPER3 = BASE / "dphil_paper_3"
COMMON = BASE / "dphil_common_cross_cutting"

OUT_DIR = PAPER3 / "results" / "threats" / "mining_risk" / "river_flood_restoration_bauxite_threats"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RIVER_EAD_MIN_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_min.tif"
RIVER_EAD_MAX_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_max.tif"

BAUXITE_RESERVES_PATH = COMMON / "common_incoming_data" / "bauxite" / "Bauxite areas.shp"
BAUXITE_BEARING_AREAS_PATH = COMMON / "common_incoming_data" / "bauxite" / "Bauxite Bearing Areas.shp"
FOREST_RESERVES_PATH = COMMON / "common_incoming_data" / "protected_landcover" / "forest_reserves.shp"
PROTECTED_AREAS_PATH = COMMON / "common_incoming_data" / "protected_landcover" / "protected_areas.shp"
JAMAICA_BOUNDARY_PATH = COMMON / "common_incoming_data" / "boundaries" / "jamaica.gpkg"
MANGROVE_PATCHES_PATH = PAPER3 / "inputs" / "forces_of_nature_mangroves" / "mangroves.shp"
MANGROVE_EAD_PATCH_TABLE_PATH = PAPER3 / "results" / "threats" / "hurricane_melissa_damage" / "mangrove_eads_hurricane_damage" / "mangrove_ead_hurricane_damage_patch_table.csv"

J2USD = 1 / 150
RASTERIZATION_RULE = "pixel centre: rasterio.features.rasterize default all_touched=False"

OUT_DIR

## Load river-flood avoided EAD rasters

The input rasters are positive damage-reduction rasters from the river-flood forest restoration analysis. Non-positive pixels are set to `NaN`, so all sums below are positive avoided EAD only.

In [ ]:
def read_positive_ead_usd(path: Path, reference: dict | None = None) -> tuple[np.ndarray, dict]:
    """Read avoided EAD raster, convert JMD to USD, and retain only positive values."""
    with rasterio.open(path) as src:
        profile = src.profile.copy()
        arr = src.read(1).astype("float64") * J2USD

    arr[~np.isfinite(arr)] = np.nan
    arr[arr <= 0] = np.nan

    if reference is not None:
        checks = {
            "crs": profile["crs"] == reference["crs"],
            "transform": profile["transform"] == reference["transform"],
            "height": profile["height"] == reference["height"],
            "width": profile["width"] == reference["width"],
        }
        if not all(checks.values()):
            raise ValueError(f"Raster alignment mismatch for {path}: {checks}")

    return arr, profile

river_ead_min_usd, ref = read_positive_ead_usd(RIVER_EAD_MIN_PATH)
river_ead_max_usd, _ = read_positive_ead_usd(RIVER_EAD_MAX_PATH, ref)

shape = (ref["height"], ref["width"])
transform = ref["transform"]
crs = ref["crs"]
pixel_area_ha = abs(transform.a * transform.e) / 10_000

min_positive = np.isfinite(river_ead_min_usd) & (river_ead_min_usd > 0)
max_positive = np.isfinite(river_ead_max_usd) & (river_ead_max_usd > 0)
benefit_mask = min_positive | max_positive

scenario_arrays = {
    "minimum": river_ead_min_usd,
    "maximum": river_ead_max_usd,
}
scenario_positive_masks = {
    "minimum": min_positive,
    "maximum": max_positive,
}
scenario_totals_usd = {
    scenario: float(np.nansum(arr)) for scenario, arr in scenario_arrays.items()
}

river_grid_summary = pd.DataFrame([
    {
        "scenario": scenario,
        "positive_pixels": int(mask.sum()),
        "positive_area_ha": float(mask.sum() * pixel_area_ha),
        "positive_avoided_ead_usd": scenario_totals_usd[scenario],
        "positive_avoided_ead_usd_million": scenario_totals_usd[scenario] / 1e6,
    }
    for scenario, mask in scenario_positive_masks.items()
])

river_grid_summary

## Rasterize bauxite and protected-area masks

Bauxite reserves and the protected-area/forest-reserve union are converted to Boolean masks on the avoided-EAD raster grid. This keeps all area and avoided-EAD summaries in the same spatial units as the river-flood results.

In [ ]:
def clean_geometries(gdf: gpd.GeoDataFrame, target_crs) -> gpd.GeoDataFrame:
    """Drop empty geometries, reproject, and repair invalid geometry where needed."""
    gdf = gdf.to_crs(target_crs)
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    gdf["geometry"] = gdf.geometry.make_valid()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    return gdf


def rasterize_gdf(gdf: gpd.GeoDataFrame, *, shape: tuple[int, int], transform) -> np.ndarray:
    shapes = [(geom, 1) for geom in gdf.geometry if geom is not None and not geom.is_empty]
    if not shapes:
        raise ValueError("No geometries available for rasterization")
    return rasterize(
        shapes,
        out_shape=shape,
        transform=transform,
        fill=0,
        dtype="uint8",
        all_touched=False,
    ).astype(bool)

bauxite_reserves = clean_geometries(gpd.read_file(BAUXITE_RESERVES_PATH), crs)
forest_reserves = clean_geometries(gpd.read_file(FOREST_RESERVES_PATH), crs)
protected_declarations = clean_geometries(gpd.read_file(PROTECTED_AREAS_PATH), crs)
protected_areas = gpd.GeoDataFrame(
    pd.concat(
        [forest_reserves[["geometry"]], protected_declarations[["geometry"]]],
        ignore_index=True,
    ),
    geometry="geometry",
    crs=crs,
)
jamaica_boundary = clean_geometries(gpd.read_file(JAMAICA_BOUNDARY_PATH), crs)

bauxite_mask = rasterize_gdf(bauxite_reserves, shape=shape, transform=transform)
protected_mask = rasterize_gdf(protected_areas, shape=shape, transform=transform)

mask_checks = pd.DataFrame([
    {
        "layer": "bauxite reserves",
        "vector_area_km2": bauxite_reserves.geometry.union_all().area / 1e6,
        "rasterized_area_km2_in_ead_grid": bauxite_mask.sum() * pixel_area_ha / 100,
    },
    {
        "layer": "protected areas and forest reserves",
        "vector_area_km2": protected_areas.geometry.union_all().area / 1e6,
        "rasterized_area_km2_in_ead_grid": protected_mask.sum() * pixel_area_ha / 100,
    },
    {
        "layer": "bauxite reserves intersecting protected union",
        "vector_area_km2": bauxite_reserves.geometry.union_all().intersection(protected_areas.geometry.union_all()).area / 1e6,
        "rasterized_area_km2_in_ead_grid": (bauxite_mask & protected_mask).sum() * pixel_area_ha / 100,
    },
])

mask_checks

## Summarise restoration benefit area and avoided EAD on bauxite

In [ ]:
def pct(numerator: float, denominator: float) -> float:
    return float(numerator / denominator * 100) if denominator else np.nan


def area_ha(mask: np.ndarray) -> float:
    return float(mask.sum() * pixel_area_ha)


def ead_sum_usd(arr: np.ndarray, mask: np.ndarray) -> float:
    valid = mask & np.isfinite(arr) & (arr > 0)
    return float(arr[valid].sum())

analysis_masks = {
    "all restoration-benefit pixels": benefit_mask,
    "on bauxite": benefit_mask & bauxite_mask,
    "on bauxite and protected": benefit_mask & bauxite_mask & protected_mask,
    "on bauxite and unprotected": benefit_mask & bauxite_mask & ~protected_mask,
    "not on bauxite": benefit_mask & ~bauxite_mask,
}

on_bauxite_union_area = area_ha(analysis_masks["on bauxite"])

union_rows = []
for group, mask in analysis_masks.items():
    rec = {
        "group": group,
        "benefit_pixels_union_min_or_max": int(mask.sum()),
        "benefit_area_ha_union_min_or_max": area_ha(mask),
        "pct_total_benefit_area_union_min_or_max": pct(area_ha(mask), area_ha(benefit_mask)),
        "pct_on_bauxite_benefit_area_union_min_or_max": np.nan,
    }
    if group in {"on bauxite", "on bauxite and protected", "on bauxite and unprotected"}:
        rec["pct_on_bauxite_benefit_area_union_min_or_max"] = pct(area_ha(mask), on_bauxite_union_area)

    for scenario, arr in scenario_arrays.items():
        scenario_positive_mask = mask & scenario_positive_masks[scenario]
        scenario_area = area_ha(scenario_positive_mask)
        ead_usd = ead_sum_usd(arr, scenario_positive_mask)
        on_bauxite_ead = ead_sum_usd(arr, analysis_masks["on bauxite"] & scenario_positive_masks[scenario])
        on_bauxite_area = area_ha(analysis_masks["on bauxite"] & scenario_positive_masks[scenario])
        rec[f"positive_area_ha_{scenario}"] = scenario_area
        rec[f"pct_total_positive_area_{scenario}"] = pct(scenario_area, area_ha(scenario_positive_masks[scenario]))
        rec[f"pct_on_bauxite_positive_area_{scenario}"] = np.nan
        if group in {"on bauxite", "on bauxite and protected", "on bauxite and unprotected"}:
            rec[f"pct_on_bauxite_positive_area_{scenario}"] = pct(scenario_area, on_bauxite_area)
        rec[f"positive_avoided_ead_usd_{scenario}"] = ead_usd
        rec[f"positive_avoided_ead_usd_million_{scenario}"] = ead_usd / 1e6
        rec[f"pct_total_positive_avoided_ead_{scenario}"] = pct(ead_usd, scenario_totals_usd[scenario])
        rec[f"pct_on_bauxite_positive_avoided_ead_{scenario}"] = np.nan
        if group in {"on bauxite", "on bauxite and protected", "on bauxite and unprotected"}:
            rec[f"pct_on_bauxite_positive_avoided_ead_{scenario}"] = pct(ead_usd, on_bauxite_ead)

    union_rows.append(rec)

overall_summary = pd.DataFrame(union_rows)
overall_summary

In [ ]:
scenario_rows = []
scenario_groups = {
    "all positive avoided EAD restoration pixels": None,
    "on bauxite": bauxite_mask,
    "on bauxite and protected": bauxite_mask & protected_mask,
    "on bauxite and unprotected": bauxite_mask & ~protected_mask,
    "not on bauxite": ~bauxite_mask,
}

for scenario, arr in scenario_arrays.items():
    scenario_positive = scenario_positive_masks[scenario]
    scenario_area_total = area_ha(scenario_positive)
    on_bauxite_positive = scenario_positive & bauxite_mask
    on_bauxite_area = area_ha(on_bauxite_positive)
    on_bauxite_ead = ead_sum_usd(arr, on_bauxite_positive)

    for group, extra_mask in scenario_groups.items():
        mask = scenario_positive if extra_mask is None else scenario_positive & extra_mask
        ead_usd = ead_sum_usd(arr, mask)
        rec = {
            "scenario": scenario,
            "group": group,
            "positive_pixels": int(mask.sum()),
            "positive_area_ha": area_ha(mask),
            "pct_scenario_positive_area": pct(area_ha(mask), scenario_area_total),
            "positive_avoided_ead_usd": ead_usd,
            "positive_avoided_ead_usd_million": ead_usd / 1e6,
            "pct_scenario_positive_avoided_ead": pct(ead_usd, scenario_totals_usd[scenario]),
            "pct_on_bauxite_positive_area": np.nan,
            "pct_on_bauxite_positive_avoided_ead": np.nan,
        }
        if group in {"on bauxite", "on bauxite and protected", "on bauxite and unprotected"}:
            rec["pct_on_bauxite_positive_area"] = pct(area_ha(mask), on_bauxite_area)
            rec["pct_on_bauxite_positive_avoided_ead"] = pct(ead_usd, on_bauxite_ead)
        scenario_rows.append(rec)

scenario_summary = pd.DataFrame(scenario_rows)
scenario_summary

## Written summary values

In [ ]:
def row(group: str) -> pd.Series:
    return overall_summary.loc[overall_summary["group"].eq(group)].iloc[0]


def fnum(value: float, decimals: int = 1) -> str:
    return f"{value:,.{decimals}f}"


def fpct(value: float, decimals: int = 1) -> str:
    return f"{value:.{decimals}f}%"


def usd_million_range(row_obj: pd.Series, prefix: str = "positive_avoided_ead_usd") -> str:
    return (
        f"US${row_obj[f'{prefix}_minimum'] / 1e6:,.2f}-"
        f"{row_obj[f'{prefix}_maximum'] / 1e6:,.2f} million"
    )


def pct_range(row_obj: pd.Series, prefix: str) -> str:
    return (
        f"{row_obj[f'{prefix}_minimum']:.1f}% and "
        f"{row_obj[f'{prefix}_maximum']:.1f}%"
    )

all_row = row("all restoration-benefit pixels")
bauxite_row = row("on bauxite")
protected_row = row("on bauxite and protected")
unprotected_row = row("on bauxite and unprotected")

summary_text = f"""# River flood forest restoration avoided EADs and bauxite reserves

Across {fnum(all_row['benefit_area_ha_union_min_or_max'])} ha of river-flood forest restoration area with positive avoided EAD in either the minimum or maximum scenario, {fnum(bauxite_row['benefit_area_ha_union_min_or_max'])} ha are located on mapped bauxite reserves, equivalent to {fpct(bauxite_row['pct_total_benefit_area_union_min_or_max'])} of the national restoration-benefit area. These bauxite-overlapping restoration pixels account for {usd_million_range(bauxite_row)}, representing {pct_range(bauxite_row, 'pct_total_positive_avoided_ead')} of total positive river-flood restoration avoided EAD in the minimum and maximum scenarios, respectively.

Of the restoration-benefit area on bauxite reserves, {fnum(protected_row['benefit_area_ha_union_min_or_max'])} ha are within the protected-area and forest-reserve union. This is {fpct(protected_row['pct_on_bauxite_benefit_area_union_min_or_max'])} of the bauxite-overlapping restoration-benefit area and {fpct(protected_row['pct_total_benefit_area_union_min_or_max'])} of all river-flood restoration-benefit area. These protected bauxite-overlapping pixels account for {usd_million_range(protected_row)}, equivalent to {pct_range(protected_row, 'pct_on_bauxite_positive_avoided_ead')} of bauxite-overlapping avoided EAD and {pct_range(protected_row, 'pct_total_positive_avoided_ead')} of total positive river-flood restoration avoided EAD.

Consequently, {fnum(unprotected_row['benefit_area_ha_union_min_or_max'])} ha, or {fpct(unprotected_row['pct_on_bauxite_benefit_area_union_min_or_max'])}, of bauxite-overlapping restoration-benefit area are outside the protected-area and forest-reserve union. These unprotected bauxite-overlapping pixels account for {usd_million_range(unprotected_row)}, or {pct_range(unprotected_row, 'pct_on_bauxite_positive_avoided_ead')} of the bauxite-overlapping avoided EAD range.
"""

display(Markdown(summary_text))

## Map restoration benefit overlap with bauxite and protection

In [ ]:
plot_arr = np.zeros(shape, dtype="uint8")
plot_arr[benefit_mask & ~bauxite_mask] = 1
plot_arr[benefit_mask & bauxite_mask & ~protected_mask] = 2
plot_arr[benefit_mask & bauxite_mask & protected_mask] = 3
plot_arr = np.ma.masked_where(plot_arr == 0, plot_arr)

bounds = rasterio.transform.array_bounds(ref["height"], ref["width"], transform)
left, bottom, right, top = bounds

colors = ["#d8d8d8", "#b76534", "#1f8a70"]
cmap = ListedColormap(colors)
norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5], cmap.N)

fig, ax = plt.subplots(figsize=(10, 6), dpi=180)
ax.imshow(
    plot_arr,
    extent=(left, right, bottom, top),
    origin="upper",
    cmap=cmap,
    norm=norm,
    interpolation="nearest",
)

jamaica_boundary.boundary.plot(ax=ax, color="black", linewidth=0.7)
bauxite_reserves.boundary.plot(ax=ax, color="#7a3d1f", linewidth=0.45, alpha=0.8)

handles = [
    mpatches.Patch(color="#d8d8d8", label="Restoration benefit area not on bauxite"),
    mpatches.Patch(color="#b76534", label="Restoration benefit area on bauxite, unprotected"),
    mpatches.Patch(color="#1f8a70", label="Restoration benefit area on bauxite, protected"),
]
ax.legend(handles=handles, loc="lower left", frameon=True, fontsize=8)
Robyn_paper_2_defs.add_scale_bar(
    ax,
    jamaica_boundary,
    where="right-bottom",
    length_km=20,
    pad=0.05,
    lw=0.8,
    fs_lab=8,
    fs_unit=8,
)
Robyn_paper_2_defs.add_north_arrow_axes(
    ax,
    0.89,
    0.78,
    size_frac=0.055,
    gap_frac=0.035,
    fs=8,
    lw=0.8,
)
ax.set_title("River-flood forest restoration benefit area on bauxite reserves")
ax.set_axis_off()
fig.tight_layout()

map_path = OUT_DIR / "river_flood_restoration_bauxite_overlap_map.png"
fig.savefig(map_path, dpi=300, bbox_inches="tight")
plt.show()

map_path

## Check mangrove avoided EAD areas against bauxite-bearing areas

This checks whether Forces of Nature mangrove patches with positive avoided EAD in either coastal-flood scenario intersect the broader mapped bauxite-bearing areas.

In [ ]:
bauxite_bearing_areas = clean_geometries(gpd.read_file(BAUXITE_BEARING_AREAS_PATH), crs)
mangrove_patches = clean_geometries(gpd.read_file(MANGROVE_PATCHES_PATH), crs)
mangrove_ead_table = pd.read_csv(MANGROVE_EAD_PATCH_TABLE_PATH)

mangrove_patches["Mangrove_ID"] = mangrove_patches["ID"].astype(int)
positive_mangrove_ids = set(
    mangrove_ead_table.loc[
        mangrove_ead_table["positive_avoided_ead_either"].astype(bool),
        "Mangrove_ID",
    ].astype(int)
)

mangrove_ead_columns = [
    "Mangrove_ID",
    "positive_avoided_ead_usd_min",
    "positive_avoided_ead_usd_max",
    "positive_avoided_ead_either",
]
all_mangroves_with_ead = mangrove_patches.merge(
    mangrove_ead_table[mangrove_ead_columns],
    on="Mangrove_ID",
    how="left",
)
mangroves_with_avoided_ead = all_mangroves_with_ead[
    all_mangroves_with_ead["Mangrove_ID"].isin(positive_mangrove_ids)
].copy()

bauxite_bearing_union = bauxite_bearing_areas.geometry.union_all()
for mangrove_group in [all_mangroves_with_ead, mangroves_with_avoided_ead]:
    mangrove_group["area_on_bauxite_bearing_ha"] = mangrove_group.geometry.intersection(bauxite_bearing_union).area / 10_000
    mangrove_group["intersects_bauxite_bearing"] = mangrove_group["area_on_bauxite_bearing_ha"] > 0

mangrove_bauxite_bearing_overlap = mangroves_with_avoided_ead.loc[
    mangroves_with_avoided_ead["intersects_bauxite_bearing"],
    [
        "Mangrove_ID",
        "Parish",
        "TYPE",
        "HECTARES",
        "area_on_bauxite_bearing_ha",
        "positive_avoided_ead_usd_min",
        "positive_avoided_ead_usd_max",
    ],
].sort_values("area_on_bauxite_bearing_ha", ascending=False)

mangrove_bauxite_bearing_summary = pd.DataFrame([
    {
        "group": "all Forces of Nature mangrove patches",
        "patch_count": len(all_mangroves_with_ead),
        "total_area_ha": all_mangroves_with_ead.geometry.area.sum() / 10_000,
        "patch_count_on_bauxite_bearing": int(all_mangroves_with_ead["intersects_bauxite_bearing"].sum()),
        "area_on_bauxite_bearing_ha": all_mangroves_with_ead["area_on_bauxite_bearing_ha"].sum(),
        "pct_area_on_bauxite_bearing": pct(
            all_mangroves_with_ead["area_on_bauxite_bearing_ha"].sum(),
            all_mangroves_with_ead.geometry.area.sum() / 10_000,
        ),
        "positive_avoided_ead_usd_min": all_mangroves_with_ead["positive_avoided_ead_usd_min"].fillna(0).sum(),
        "positive_avoided_ead_usd_max": all_mangroves_with_ead["positive_avoided_ead_usd_max"].fillna(0).sum(),
    },
    {
        "group": "mangrove patches with positive avoided EAD in either scenario",
        "patch_count": len(mangroves_with_avoided_ead),
        "total_area_ha": mangroves_with_avoided_ead.geometry.area.sum() / 10_000,
        "patch_count_on_bauxite_bearing": int(mangroves_with_avoided_ead["intersects_bauxite_bearing"].sum()),
        "area_on_bauxite_bearing_ha": mangroves_with_avoided_ead["area_on_bauxite_bearing_ha"].sum(),
        "pct_area_on_bauxite_bearing": pct(
            mangroves_with_avoided_ead["area_on_bauxite_bearing_ha"].sum(),
            mangroves_with_avoided_ead.geometry.area.sum() / 10_000,
        ),
        "positive_avoided_ead_usd_min": mangroves_with_avoided_ead["positive_avoided_ead_usd_min"].fillna(0).sum(),
        "positive_avoided_ead_usd_max": mangroves_with_avoided_ead["positive_avoided_ead_usd_max"].fillna(0).sum(),
    },
])

mangrove_bauxite_bearing_summary

In [ ]:
all_mangrove_summary = mangrove_bauxite_bearing_summary.loc[
    mangrove_bauxite_bearing_summary["group"].eq("all Forces of Nature mangrove patches")
].iloc[0]
positive_mangrove_summary = mangrove_bauxite_bearing_summary.loc[
    mangrove_bauxite_bearing_summary["group"].eq("mangrove patches with positive avoided EAD in either scenario")
].iloc[0]

mangrove_bauxite_bearing_summary_text = f"""# Mangrove avoided EAD patches and bauxite-bearing areas

No Forces of Nature mangrove patches with positive avoided EAD in either coastal-flood scenario intersect mapped bauxite-bearing areas. This check covered {int(positive_mangrove_summary['patch_count'])} positive avoided-EAD mangrove patches covering {fnum(positive_mangrove_summary['total_area_ha'])} ha and representing US${positive_mangrove_summary['positive_avoided_ead_usd_min'] / 1e6:,.2f}-{positive_mangrove_summary['positive_avoided_ead_usd_max'] / 1e6:,.2f} million in positive avoided EAD.

The result is also zero when checking all {int(all_mangrove_summary['patch_count'])} mapped Forces of Nature mangrove patches, not only the benefit-providing subset.
"""

display(Markdown(mangrove_bauxite_bearing_summary_text))
mangrove_bauxite_bearing_overlap

## Write outputs

In [ ]:
metadata = pd.DataFrame([
    {"name": "river_ead_min_path", "value": str(RIVER_EAD_MIN_PATH)},
    {"name": "river_ead_max_path", "value": str(RIVER_EAD_MAX_PATH)},
    {"name": "bauxite_reserves_path", "value": str(BAUXITE_RESERVES_PATH)},
    {"name": "bauxite_bearing_areas_path", "value": str(BAUXITE_BEARING_AREAS_PATH)},
    {"name": "forest_reserves_path", "value": str(FOREST_RESERVES_PATH)},
    {"name": "protected_areas_path", "value": str(PROTECTED_AREAS_PATH)},
    {"name": "jamaica_boundary_path", "value": str(JAMAICA_BOUNDARY_PATH)},
    {"name": "mangrove_patches_path", "value": str(MANGROVE_PATCHES_PATH)},
    {"name": "mangrove_ead_patch_table_path", "value": str(MANGROVE_EAD_PATCH_TABLE_PATH)},
    {"name": "raster_crs", "value": str(crs)},
    {"name": "raster_shape", "value": str(shape)},
    {"name": "pixel_area_ha", "value": pixel_area_ha},
    {"name": "currency_conversion", "value": "JMD to USD = 1/150"},
    {"name": "river_benefit_area_definition", "value": "positive avoided EAD in minimum or maximum river-flood restoration scenario"},
    {"name": "mangrove_benefit_area_definition", "value": "Forces of Nature mangrove patches with positive avoided EAD in minimum or maximum coastal-flood scenario"},
    {"name": "rasterization_rule", "value": RASTERIZATION_RULE},
])

outputs = {
    "river_grid_summary": OUT_DIR / "river_flood_restoration_bauxite_overlap_grid_summary.csv",
    "mask_checks": OUT_DIR / "river_flood_restoration_bauxite_overlap_mask_checks.csv",
    "overall_summary": OUT_DIR / "river_flood_restoration_bauxite_overlap_overall_summary.csv",
    "scenario_summary": OUT_DIR / "river_flood_restoration_bauxite_overlap_scenario_summary.csv",
    "mangrove_bauxite_bearing_summary": OUT_DIR / "mangrove_avoided_ead_bauxite_bearing_summary.csv",
    "mangrove_bauxite_bearing_overlap": OUT_DIR / "mangrove_avoided_ead_bauxite_bearing_overlap_patches.csv",
    "method_metadata": OUT_DIR / "river_flood_restoration_bauxite_overlap_method_metadata.csv",
}

river_grid_summary.to_csv(outputs["river_grid_summary"], index=False)
mask_checks.to_csv(outputs["mask_checks"], index=False)
overall_summary.to_csv(outputs["overall_summary"], index=False)
scenario_summary.to_csv(outputs["scenario_summary"], index=False)
mangrove_bauxite_bearing_summary.to_csv(outputs["mangrove_bauxite_bearing_summary"], index=False)
mangrove_bauxite_bearing_overlap.to_csv(outputs["mangrove_bauxite_bearing_overlap"], index=False)
metadata.to_csv(outputs["method_metadata"], index=False)

summary_path = OUT_DIR / "river_flood_restoration_bauxite_overlap_written_summary.md"
summary_path.write_text(summary_text)

mangrove_summary_path = OUT_DIR / "mangrove_avoided_ead_bauxite_bearing_written_summary.md"
mangrove_summary_path.write_text(mangrove_bauxite_bearing_summary_text)

list(outputs.values()) + [summary_path, mangrove_summary_path, map_path]